In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:

import torch
from fvcore.nn import FlopCountAnalysis, flop_count_table
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

from admmtor.eprocessing.dataload import ImageDataset
from admmtor.modelbuild.denoiser import DivergentRestorer
from admmtor.eprocessing.etransforms import Scale, AddAWGN

In [ ]:
sns.set_palette("Spectral")

In [ ]:
from enum import Enum

class AnalysisMetric(Enum):
	OBJECTIVE_FIDELITY = ['psnr', 'ssim', 'msssim', 'scc', 'uiq', 'i-sam']
	SUBJECTIVE_FIDELITY = ['arniqa', 'i-lpips', 'i-dists', 'i-bfe']

In [ ]:
from prettytable import PrettyTable

def get_model_params(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        try:
            params = parameter.numel()
            table.add_row([name, params])
            total_params += params
        except Exception as e:
            print(f"Error processing parameter {name}: {e}")
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params

In [ ]:
def average_psnr(mse_values, max_pixel_value=1.0):
    psnr_values = []
    for mse in mse_values:
        if mse == 0:
            psnr = float('inf')  # Infinite PSNR for zero MSE
        else:
            psnr = 10 * np.log10((max_pixel_value ** 2) / np.mean(mse))
        psnr_values.append(psnr)
    
    return np.mean(psnr_values)

In [ ]:
def geometric_mean(values):
    product = 1.0
    n = len(values)
    for value in values:
        product *= value
    return product ** (1/n) if n > 0 else 0

In [ ]:
def get_flop_count(model, device='cuda'):
    dummy_input = torch.randn(1, 3, 224, 224).to(device)  # Adjust the shape according to your model's input
    flops = FlopCountAnalysis(model, dummy_input)
    return f"{flops.total() / 1e9:.2f} GFLOPs"

# Analyze ADMM-TV layer outputs

In [ ]:
DECONV1 = {'kern_size': (),
         'max_iters': 100,
         'iso': True}
DECONV2 = {'kern_size': (),
         'max_iters': 100,
         'iso': True}

device = 'cuda'
modelp = Path('D:/Projects/torch-admm-deconv/trained_models/2026/blind/admm/admm-bsd-div2k-15and35std-june-128-cwa-modif-2_epoch04_vloss0.1031.tar')
# FROZE = DivergentRestorer(3, 2, 3,
#                           3, 4, 86,
#                           86, 8,
#                           output_activation=torch.nn.Sigmoid(), admms=[DECONV1, DECONV2])
ADMM = DivergentRestorer([2, 8, 32], 3,
                              3, 86,
                              86, 8,
                              output_activation=torch.nn.Sigmoid(), admms=[DECONV1, DECONV2])
model_d = torch.load(modelp, weights_only=False)
ADMM.load_state_dict(model_d['model_state_dict'])
ADMM = ADMM.to(device)
ADMM = ADMM.eval()

In [ ]:
feature_extractor = create_feature_extractor(ADMM, return_nodes=['blocks.0.admms.0', 'blocks.0.admms.1'])

In [ ]:
min_std, max_std = 25, 26
xp = Path(r'D:\\Projects\\datasets\\denoising_test_sets\\CBSD68')
yp = xp

In [ ]:
imd = ImageDataset(xp, yp, transforms=[Scale(), AddAWGN(std_range=(min_std, max_std), both=False)])
im_loader = torch.utils.data.DataLoader(imd, shuffle=False, batch_size=1)

In [ ]:
x_test, y_test = next(iter(im_loader))
features = feature_extractor(x_test.to(device))
feat_admm0 = features['blocks.0.admms.0']
feat_admm1 = features['blocks.0.admms.1']

In [ ]:
# Plot the first 3 CBSD68 images and their ADMM-TV outputs in one figure
fig, axes = plt.subplots(3, 3, figsize=(12, 15), gridspec_kw={"wspace": 0.02, "hspace": 0.15})

for row, (noisy_img, clean_img, name) in enumerate(dataset_pairs[:3]):
	noisy_img = np.asarray(noisy_img, dtype=np.float32)
	clean_img = np.asarray(clean_img, dtype=np.float32)

	with torch.no_grad():
		noisy_t = torch.from_numpy(noisy_img).permute(2, 0, 1).unsqueeze(0).to(device)
		features = feature_extractor(noisy_t)
		layer1_img = features["blocks.0.admms.0"][0].cpu().numpy().transpose(1, 2, 0)
		layer2_img = features["blocks.0.admms.1"][0].cpu().numpy().transpose(1, 2, 0)

	plots = [
		(noisy_img, "Noisy Image"),
		(layer1_img, "ADMM TV Layer 1 Output"),
		(layer2_img, "ADMM TV Layer 2 Output"),
	]

	for ax, (img, title) in zip(axes[row], plots):
		mse = np.mean((img - clean_img) ** 2)
		ax.imshow(normalize_for_display(img))
		ax.set_title(f"{title}\nMSE: {mse:.6f}", fontsize=10, fontweight="bold")
		ax.axis("off")

	axes[row, 0].set_ylabel(name, fontsize=13, fontweight="bold", rotation=90, labelpad=15)

fig.suptitle("First 3 CBSD68 Images: Noisy Inputs and post-training ADMM-TV Layer Outputs",
			 fontsize=16, fontweight="bold", y=0.92)

plt.savefig("admm_tv_layer_outputs_first_3_images.png", dpi=300,
			transparent=True, bbox_inches="tight", pad_inches=0)
plt.show()


In [ ]:
# Vizualize the rgb image
plt.figure(figsize=(10, 10))
plt.imshow(feat_admm1.detach().cpu().numpy()[0].transpose(1, 2, 0))

# Analyze test outcomes

In [ ]:
import re

def load_metrics_summary(cmode, keep=None, weighting='image', exclude_dsets=(),
                         root='D:/Projects/datasets/denoising_test_sets'):
    """Load the per-image metric CSVs of every method and average them.

    Args:
        cmode: 'gray', 'color', or 'both' (gray and color results averaged together).
            With 'both', a metric computed only on color images (e.g. lpips, sam, arniqa)
            is averaged over the color results alone.
        keep: which factor to keep separate, averaging over the others:
            None      -> one row per model (avg over color modes, datasets and stds)
            'std'     -> one row per (model, std)
            'dataset' -> one row per (model, dataset)
            'cmode'   -> one row per (model, cmode)
        weighting: how images are weighted when averaging over datasets:
            'image'   -> pool all images, so larger datasets count for more
            'dataset' -> average each dataset first, so all datasets count equally
        exclude_dsets: dataset names to leave out, e.g. ['Kodak24'].
        root: directory holding the gray/ and color/ results.

    Returns:
        DataFrame indexed by model (plus `keep`, if given), one column per metric.
    """
    if cmode not in ('gray', 'color', 'both'):
        raise ValueError(f"cmode must be 'gray', 'color' or 'both', got {cmode!r}")
    if keep not in (None, 'std', 'dataset', 'cmode'):
        raise ValueError(f"keep must be None, 'std', 'dataset' or 'cmode', got {keep!r}")
    if weighting not in ('image', 'dataset'):
        raise ValueError(f"weighting must be 'image' or 'dataset', got {weighting!r}")

    cmodes = ('gray', 'color') if cmode == 'both' else (cmode,)
    frames = []
    for mode in cmodes:
        for csv_path in (Path(root) / mode).glob('*_models_results_*std/*/metrics_*.csv'):
            # e.g. CBSD68_models_results_15_16std/admm/metrics_admm.csv
            dataset, std = re.fullmatch(r'(.+)_models_results_(\d+)_\d+std', csv_path.parent.parent.name).groups()
            if dataset not in exclude_dsets:
                frames.append(pd.read_csv(csv_path).assign(
                    model=csv_path.parent.name, cmode=mode, dataset=dataset, std=int(std)))
    if not frames:
        raise FileNotFoundError(f'No metric CSVs found for {cmode!r} under {root}')

    df = pd.concat(frames, ignore_index=True)
    id_cols = ('model', 'cmode', 'dataset', 'std')
    metrics = [col for col in df.columns if col not in id_cols]
    if weighting == 'dataset':
        df = df.groupby(list(id_cols), as_index=False)[metrics].mean()
    # mean() skips missing values, so color-only metrics are averaged over the color rows only
    return df.groupby(['model'] if keep is None else ['model', keep])[metrics].mean()

In [ ]:
work_df = load_metrics_summary(cmode='color')
work_df['i-sam'] = 1 - work_df['sam']
work_df['i-lpips'] = 1 - work_df['lpips']
work_df['i-dists'] = 1 - work_df['dists']
work_df['i-bfe'] = 1 - work_df['blur_effect']
work_df = work_df[AnalysisMetric.OBJECTIVE_FIDELITY.value + AnalysisMetric.SUBJECTIVE_FIDELITY.value]

## Prepare data

In [ ]:
cmode = 'color'

In [ ]:
dsets_p = f'D:/Projects/datasets/denoising_test_sets/{cmode}'
res_paths = [res_p for res_p in Path(dsets_p).glob('*std*')]

In [ ]:
models_names = ['bm3d', 'dncnn', 'ffdnet', 'restormer', 'dranet', 'restormer', 'swinir', 'nafnet', 'admm']

In [ ]:
res_paths_wmodels = [f'{path}/{model}/metrics_{model}.csv'.replace("\\", "/") for path in res_paths for model in models_names]

In [ ]:
res_paths_wmodels = [fname for fname in res_paths_wmodels]

In [ ]:
results_dfs = []
for path in res_paths_wmodels:
    df = pd.read_csv(path)
    parts = path.split("/")
    model_name = parts[-1].split("_")[-1].strip('.csv')
    if model_name == 'winir':
        model_name = 'swinir'
    df["model_name"] = model_name
    std_val = parts[-3].split("_")[-2]
    df["std_val"] = std_val
    dset_name = parts[-3].split("_")[0]
    df["dset_name"] = dset_name 
    results_dfs.append(df)
all_dfs = pd.concat(results_dfs, axis=0).reset_index(drop=True)

In [ ]:
if cmode == 'color':
	all_dfs['i-sam'] = 1 - all_dfs['sam']
	all_dfs['i-dists'] = 1 - all_dfs['dists']
	all_dfs['i-lpips'] = 1 - all_dfs['lpips']
	all_dfs['i-bfe'] = 1 - all_dfs['blur_effect']
else:
	all_dfs['i-dists'] = 1 - all_dfs['dists']
	all_dfs['i-bfe'] = 1 - all_dfs['blur_effect']

In [ ]:
metrics_n = [col for col in all_dfs.columns if col not in ['model_name', 'std_val', 'dset_name']]

In [ ]:
barplot_model_metrics_dict = {}
for model in models_names:
    barplot_model_metrics_dict[model] = []
    for metric in metrics_n:
        if metric == 'psnr':
            mse_values = all_dfs.loc[all_dfs['model_name'] == model, 'mse'].values
            barplot_model_metrics_dict[model].append(average_psnr(mse_values))
        else:
            barplot_model_metrics_dict[model].append(np.mean(all_dfs.loc[all_dfs['model_name'] == model, metric].values))
barplot_df = pd.DataFrame(barplot_model_metrics_dict, index=metrics_n).T

In [ ]:
barplot_df

## Plot Stats

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator


def plot_grouped_metrics(df, metrics_subset, normalize=False, index_col=0):
	if not isinstance(metrics_subset, AnalysisMetric):
		raise ValueError("metrics_subset must be an AnalysisMetric member.")

	column_aliases = {
		"i-sam": "isam",
		"i-dists": "idists",
		"i-lpips": "ilpips",
		"i-bfe": "ibfe",
	}

	# Colorblind-checked categorical order: each metric keeps its color across both panels
	series_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
	ink, secondary_ink, muted_ink = "#0b0b0b", "#52514e", "#898781"
	grid_color, baseline_color = "#e1e0d9", "#c3c2b7"
	ours_label = "DR (OURS)"

	metrics = metrics_subset.value

	if not metrics:
		raise ValueError("None of the selected metrics are present in the dataframe.")

	metric_colors = dict(zip(metrics, series_colors))

	plot_df = df[metrics].copy()
	plot_df["Model"] = [ours_label if name == "admm" else str(name).upper() for name in plot_df.index]

	has_psnr = "psnr" in metrics
	other_metrics = [metric for metric in metrics if metric != "psnr"]

	# keep original (non-normalized) values for annotations
	raw_df = df[metrics].copy()

	if normalize and other_metrics:
		denominator = plot_df[other_metrics].max() - plot_df[other_metrics].min()
		plot_df[other_metrics] = (
			plot_df[other_metrics] - plot_df[other_metrics].min()
		) / denominator.replace(0, 1)

	# panels sit side by side, with PSNR as a narrow column next to the grouped metrics
	ncols = int(has_psnr) + int(bool(other_metrics))
	fig, axes = plt.subplots(
		1, ncols,
		figsize=(max(13, len(plot_df) * 1.6), 4.4),
		squeeze=False,
		gridspec_kw={"width_ratios": [1, 2.7] if ncols == 2 else [1]},
	)
	axes = axes.flatten()

	def bar_tip(bar, horizontal):
		if horizontal:
			return bar.get_width(), bar.get_y() + bar.get_height() / 2
		return bar.get_x() + bar.get_width() / 2, bar.get_height()

	def annotate_bar(ax, bar, text, is_best, horizontal):
		offset = 10 if is_best else 4
		ax.annotate(
			text,
			bar_tip(bar, horizontal),
			xytext=(offset, 0) if horizontal else (0, offset),
			textcoords="offset points",
			ha="left" if horizontal else "center",
			va="center" if horizontal else "bottom",
			# vertical bars are grouped, so their labels turn to stay within their own bar
			rotation=0 if horizontal else 90,
			fontsize=9,
			fontweight="bold" if is_best else "normal",
			color=ink if is_best else secondary_ink,
		)

	def draw_metrics(ax, selected_metrics, ylabel, title):
		horizontal = len(selected_metrics) == 1  # a single metric (PSNR) is drawn as horizontal bars
		melted_df = plot_df.melt(
			id_vars="Model",
			value_vars=selected_metrics,
			var_name="Metric",
			value_name="Score",
		)

		sns.barplot(
			data=melted_df,
			x="Score" if horizontal else "Model",
			y="Model" if horizontal else "Score",
			orient="h" if horizontal else "v",
			hue="Metric",
			hue_order=selected_metrics,
			palette=metric_colors,
			saturation=1,
			width=0.6 if horizontal else 0.8,
			edgecolor="white",
			linewidth=1.5,
			ax=ax,
		)

		for metric, container in zip(selected_metrics, ax.containers):
			value_format = ".2f" if metric == "psnr" else ".3g"
			best_model = raw_df[metric].idxmax()
			best_bar = list(plot_df.index).index(best_model)
			bar = container[best_bar]

			ax.scatter(
				*bar_tip(bar, horizontal),
				marker="*",
				s=95,
				color="gold",
				edgecolor=ink,
				linewidth=0.6,
				zorder=5,
			)

			# annotate with the raw (non-normalized) numerical value
			annotate_bar(ax, bar, format(raw_df.loc[best_model, metric], value_format),
						 is_best=True, horizontal=horizontal)

			# on a single-metric panel, also show how our model compares with the best
			if horizontal and "admm" in plot_df.index and best_model != "admm":
				ours_bar = container[list(plot_df.index).index("admm")]
				annotate_bar(ax, ours_bar, format(raw_df.loc["admm", metric], value_format),
							 is_best=False, horizontal=horizontal)

		ax.set_title(title, fontsize=13, fontweight="bold", loc="left", color=ink, pad=12)
		ax.set_xlabel(ylabel if horizontal else "", fontsize=10, color=secondary_ink)
		ax.set_ylabel("" if horizontal else ylabel, fontsize=10, color=secondary_ink)
		if horizontal:
			ax.set_xlim(0, melted_df["Score"].max() * 1.25)  # room for the value labels
			ax.xaxis.set_major_locator(MultipleLocator(10))
		else:
			ax.set_ylim(0, melted_df["Score"].max() * 1.28)
			ax.set_yticks(np.arange(0, 1.01, 0.2))  # the space above 1.0 only holds labels

		# recessive chrome: hairlines along the value axis only, so the bars stay the focus
		value_axis, category_axis = ("x", "y") if horizontal else ("y", "x")
		ax.set_axisbelow(True)
		ax.grid(axis=value_axis, color=grid_color, linewidth=0.8)
		ax.grid(axis=category_axis, visible=False)
		ax.tick_params(axis=value_axis, length=0, labelsize=9, labelcolor=muted_ink)
		ax.tick_params(axis=category_axis, length=0, labelsize=9, labelcolor=secondary_ink)
		sns.despine(ax=ax, left=not horizontal, bottom=horizontal)
		ax.spines["left" if horizontal else "bottom"].set_color(baseline_color)

		for tick_label in (ax.get_yticklabels() if horizontal else ax.get_xticklabels()):
			if tick_label.get_text() == ours_label:
				tick_label.set_fontweight("bold")
				tick_label.set_color(ink)

		# a lone metric is named by its title; the grouped panel's legend also explains the star
		if horizontal:
			ax.get_legend().remove()
			return

		handles, labels = ax.get_legend_handles_labels()
		star_key = Line2D([], [], linestyle="none", marker="*", markersize=10,
						  markerfacecolor="gold", markeredgecolor=ink, markeredgewidth=0.6)
		ax.legend(
			handles + [star_key],
			[label.upper() for label in labels] + ["Best"],
			loc="lower right",
			bbox_to_anchor=(1, 1.01),
			ncol=len(handles) + 1,
			frameon=False,
			fontsize=9,
			labelcolor=secondary_ink,
			handlelength=1,
			handleheight=1,
			handletextpad=0.5,
			columnspacing=1.2,
			borderaxespad=0,
		)

	axis_idx = 0

	if has_psnr:
		draw_metrics(
			axes[axis_idx],
			["psnr"],
			"PSNR (dB)",
			"Average PSNR",
		)
		axis_idx += 1

	if other_metrics:
		draw_metrics(
			axes[axis_idx],
			other_metrics,
			"Normalized Score" if normalize else "Score",
			"Objective metrics" if has_psnr else "Perceptual metrics",
		)

	fig.tight_layout(w_pad=3)
	plt.show()

In [ ]:
plot_grouped_metrics(barplot_df, AnalysisMetric.OBJECTIVE_FIDELITY)

In [ ]:
plot_grouped_metrics(barplot_df, AnalysisMetric.SUBJECTIVE_FIDELITY)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

df = barplot_df

higher_is_better = ['psnr', 'ssim', 'msssim', 'scc', 'uiq']
lower_is_better = ['mse',  'sam', 'blur_effect', 'dists', 'lpips']

sns.set_theme(style="whitegrid")

def scatter_models(df, xcol, ycol):
    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=df, x=xcol, y=ycol, s=250, hue=df.index, palette='tab10', legend=False)

    for i, txt in enumerate(df.index):
        plt.annotate(txt, (df[xcol].iloc[i], df[ycol].iloc[i]), 
                     xytext=(10, -5), textcoords='offset points', fontsize=12, weight='bold')

    plt.title('Perception-Distortion Trade-off', fontsize=16, fontweight='bold')
    plt.xlabel(f'1 - {xcol.upper()} (Perceptual Quality) ← Higher is Better', fontsize=12)
    plt.ylabel(f'{ycol.upper()} (Signal Fidelity) ↑ Higher is Better', fontsize=12)
    plt.tight_layout()
    plt.savefig('1_tradeoff_plot.png', dpi=300)
    plt.show()
    
    
def scatter_models_bubble(df, xcol, ycol, zcol):
    plt.figure(figsize=(10, 7))
    ax = sns.scatterplot(
        data=df, x=xcol, y=ycol, size=zcol, 
        sizes=(150, 1000), hue=df.index, palette='tab10', alpha=0.8
    )

    for i, txt in enumerate(df.index):
        plt.annotate(txt, (df[xcol].iloc[i], df[ycol].iloc[i]), 
                     xytext=(18, -5), textcoords='offset points', fontsize=12, weight='bold')

    plt.title('Perception-Distortion Trade-off (Bubble Chart)', fontsize=16, fontweight='bold')
    plt.xlabel(f'1 - {xcol.upper()} (Perceptual Quality) ← Higher is Better', fontsize=12)
    plt.ylabel(f'{ycol.upper()} (Signal Fidelity) ↑ Higher is Better', fontsize=12)
    
    # --- FIX: Filter out "None" and the size bubbles ---
    handles, labels = ax.get_legend_handles_labels()
    
    # Get the list of actual model names as strings
    model_names = df.index.astype(str).tolist()
    
    filtered_handles = []
    filtered_labels = []
    
    # Only keep the legend items that perfectly match a model name
    for h, l in zip(handles, labels):
        if l in model_names:
            filtered_handles.append(h)
            filtered_labels.append(l)
            
    plt.legend(filtered_handles, filtered_labels, 
               bbox_to_anchor=(1.05, 1), loc='upper left', title="Models")
    # ----------------------------------------------------

    plt.tight_layout()
    plt.savefig('1_tradeoff_bubble_plot.png', dpi=300)
    plt.show()
    
    
def scatter_models_3d(df, xcol, ycol, zcol):
    """
    Creates a true 3D scatter plot with X, Y, and Z axes.
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    colors = plt.cm.tab10.colors
    
    # Iterate through models to plot points and annotations
    for i, (idx, row) in enumerate(df.iterrows()):
        ax.scatter(row[xcol], row[ycol], row[zcol], s=150, color=colors[i % len(colors)], label=idx)
        
        # Add a tiny Z-offset to the text so it hovers above the point
        z_offset = (df[zcol].max() - df[zcol].min()) * 0.05
        ax.text(row[xcol], row[ycol], row[zcol] + z_offset, idx, fontsize=10, weight='bold')

    ax.set_title('Perception-Distortion Trade-off (3D)', fontsize=16, fontweight='bold', pad=20)
    
    # Label the axes
    # Make more space to display full metric name
    ax.set_xlabel(f'{xcol.upper()}')
    ax.set_ylabel(f'{ycol.upper()}')
    ax.set_zlabel(f'{zcol.upper()}', labelpad=0, rotation=70)
    
    plt.tight_layout()
    plt.savefig('1_tradeoff_3D_plot.png', dpi=300)
    plt.show()

In [ ]:
import matplotlib.patheffects as pe

# metrics where a lower value is better; every other metric counts as higher-is-better
LOWER_IS_BETTER = {'mse', 'sam', 'blur_effect', 'dists', 'lpips'}
# drawn left to right, in the order defined by AnalysisMetric
BUMP_METRIC_GROUPS = {
	'Fidelity': AnalysisMetric.OBJECTIVE_FIDELITY.value,
	'Perceptual': AnalysisMetric.SUBJECTIVE_FIDELITY.value,
}
BUMP_METRIC_LABELS = {'msssim': 'MS-SSIM', 'blur_effect': 'BLUR'}
# categorical palette in fixed order (blue, orange, aqua, yellow, magenta, green, violet, red);
# a marker shape per model keeps lines apart where hues are close for color-blind readers
BUMP_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
BUMP_MARKERS = ['o', 's', 'D', '^', 'v', 'P', 'X', 'h']


def bump_chart_ranking(df, groups=BUMP_METRIC_GROUPS, highlight=('admm',), save_path=None):
	"""Rank every model on each metric (1 = best) and draw the ranks as a bump chart.

	Args:
		df: one row per model, one column per metric, e.g. load_metrics_summary(cmode='color').
		groups: {group name: [metrics]}, drawn left to right; metrics missing from df are skipped.
		highlight: models drawn thicker, with their rank inside each marker; they take the first colors.
		save_path: optional path to save the figure to.
	"""
	ink, secondary_ink, muted_ink = '#0b0b0b', '#52514e', '#898781'
	grid_color, overflow_color = '#e1e0d9', '#bdbcb2'

	groups = {name: [m for m in ms if m in df.columns] for name, ms in groups.items()}
	groups = {name: ms for name, ms in groups.items() if ms}
	metrics = [m for ms in groups.values() for m in ms]
	if not metrics:
		raise ValueError('None of the requested metrics are in the dataframe.')

	ranks = pd.DataFrame({m: df[m].rank(ascending=m in LOWER_IS_BETTER, method='min') for m in metrics})
	n_models = len(df)

	def display_name(model):
		return 'DR (OURS)' if model == 'admm' else str(model).upper()

	# x positions: a small gap between metric groups
	xs, x, group_spans = {}, 0.0, []
	for gi, (name, ms) in enumerate(groups.items()):
		if gi:
			x += 0.5
		start = x
		for m in ms:
			xs[m] = x
			x += 1
		group_spans.append((name, start, x - 1))
	metric_x = [xs[m] for m in metrics]

	fig, ax = plt.subplots(figsize=(13, 0.55 * n_models + 2.2))
	fig.patch.set_facecolor('white')
	ax.set_facecolor('white')

	highlighted = [m for m in highlight if m in df.index]
	others = [m for m in df.index if m not in highlighted]
	# colors follow the model, not its rank; models beyond the palette fall back to gray (never cycled)
	style = {m: (BUMP_COLORS[i], BUMP_MARKERS[i]) if i < len(BUMP_COLORS) else (overflow_color, 'o')
			 for i, m in enumerate(highlighted + others)}

	for model in others + highlighted:  # highlighted lines on top
		is_hl = model in highlighted
		color, marker = style[model]
		y = ranks.loc[model, metrics].values
		lw, z = (3.2, 4) if is_hl else (2, 2)

		ax.plot(metric_x, y, color=color, lw=lw, zorder=z, solid_joinstyle='round',
				path_effects=[pe.Stroke(linewidth=lw + 3, foreground='white'), pe.Normal()])
		if is_hl:
			ax.scatter(metric_x, y, s=330, color=color, edgecolor='white', linewidth=2, zorder=z + 1)
			for px, py in zip(metric_x, y):
				ax.text(px, py, f'{int(py)}', ha='center', va='center', fontsize=8.5,
						fontweight='bold', color='white', zorder=z + 2)
		else:
			ax.scatter(metric_x, y, s=75, marker=marker, color=color, edgecolor='white', linewidth=1.2,
					   zorder=z + 1)

	# names at both line ends, at their rank on the first / last metric (tied models share a label)
	for metric, dx, ha in [(metrics[0], -0.45, 'right'), (metrics[-1], 0.45, 'left')]:
		col = ranks[metric]
		for r, models in col.groupby(col).groups.items():
			is_hl = any(m in highlighted for m in models)
			ax.text(xs[metric] + dx, r, ', '.join(display_name(m) for m in models), ha=ha, va='center',
					fontsize=10, fontweight='bold' if is_hl else 'normal', color=ink if is_hl else secondary_ink)

	# x axis on top: metric names with their direction
	ax.xaxis.tick_top()
	ax.set_xticks(metric_x)
	ax.set_xticklabels(
		[f"{BUMP_METRIC_LABELS.get(m, m.upper())} {'↓' if m in LOWER_IS_BETTER else '↑'}" for m in metrics],
		fontsize=10, color=secondary_ink)
	ax.tick_params(axis='x', length=0, pad=8)

	# y axis: rank 1 at the top; ranks are read off the markers and labels, so no tick labels
	ax.set_ylim(n_models + 0.6, 0.4)
	ax.set_yticks(range(1, n_models + 1))
	ax.tick_params(axis='y', length=0, labelleft=False)
	ax.set_xlim(metric_x[0] - 0.5, metric_x[-1] + 0.5)

	# recessive chrome: faint rank rows, dashed hairlines between groups, no spines
	ax.grid(axis='y', color=grid_color, lw=0.8)
	ax.grid(axis='x', visible=False)
	ax.set_axisbelow(True)
	for spine in ax.spines.values():
		spine.set_visible(False)
	for (_, _, end), (_, start, _) in zip(group_spans, group_spans[1:]):
		ax.axvline((end + start) / 2, color=grid_color, lw=1, ls='--', zorder=0)

	# group headers above the metric names
	header_y = 1 + 0.75 / fig.get_figheight()
	for name, start, end in group_spans:
		ax.plot([start - 0.3, end + 0.3], [header_y, header_y], color=muted_ink, lw=1,
				transform=ax.get_xaxis_transform(), clip_on=False)
		ax.text((start + end) / 2, header_y + 0.01, name.upper(), transform=ax.get_xaxis_transform(),
				ha='center', va='bottom', fontsize=10, fontweight='bold', color=secondary_ink)

	ax.text(0, header_y + 0.95 / fig.get_figheight(), 'Rank consistency across metrics',
			transform=ax.transAxes, fontsize=16, fontweight='bold', color=ink, ha='left', va='bottom')
	any_lower = any(m in LOWER_IS_BETTER for m in metrics)
	notes = ['1 = best', '↑/↓ higher/lower value is better' if any_lower else '↑ higher value is better',
			 'ties share the best rank']
	if any(m.startswith('i-') for m in metrics):
		notes.insert(2, 'I-X = 1 − X')
	ax.text(0, header_y + 0.55 / fig.get_figheight(), '  ·  '.join(notes),
			transform=ax.transAxes, fontsize=10, color=muted_ink, ha='left', va='bottom')

	if save_path:
		fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
	plt.show()

In [ ]:
bump_chart_ranking(work_df, save_path='1_bump_chart.svg')

# Qualitative visualization analysis

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def plot_image_comparison_grid(images_dict, patch_size, crop_origin, nrows=2, ncols=5,
                               highlight='admm', save_path=None, dpi=300):
    """Show the same crop from every image side by side, for a paper figure.

    Args:
        images_dict: {name: PIL image}, plotted in insertion order (e.g. clean, noisy, methods...).
        patch_size: int or (w, h) of the crop, in pixels.
        crop_origin: (x, y) top-left corner of the crop.
        nrows, ncols: grid shape; must hold every image.
        highlight: key of our method, drawn with an accent frame and a bold label.
        save_path: optional output path (.png, .pdf, .svg, ...).
    """
    ink, secondary_ink = '#0b0b0b', '#52514e'
    frame_color, accent = '#c3c2b7', '#2a78d6'

    if len(images_dict) > nrows * ncols:
        raise ValueError(f'{len(images_dict)} images do not fit in a {nrows}x{ncols} grid')

    w, h = (patch_size, patch_size) if isinstance(patch_size, int) else patch_size
    x, y = crop_origin
    patches = {name: np.asarray(img.convert('RGB').crop((x, y, x + w, y + h)))
               for name, img in images_dict.items()}

    def display_name(name):
        return {'admm': 'DR (OURS)', 'clean': 'GROUND TRUTH'}.get(name, name.upper())

    tile = 2.3  # inches per patch
    fig, axes = plt.subplots(nrows, ncols, figsize=(tile * ncols * w / h, (tile + 0.35) * nrows),
                             gridspec_kw={'wspace': 0.06, 'hspace': 0.14}, squeeze=False)
    fig.patch.set_facecolor('white')

    for ax, (name, patch) in zip(axes.flat, patches.items()):
        is_ours = name == highlight
        ax.imshow(patch, interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(accent if is_ours else frame_color)
            spine.set_linewidth(2.5 if is_ours else 0.8)

        ax.text(0.5, -0.05, display_name(name), transform=ax.transAxes, ha='center', va='top',
                fontsize=11, fontweight='bold' if is_ours else 'normal', color=ink if is_ours else secondary_ink)

    for ax in axes.flat[len(patches):]:
        ax.axis('off')

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight', pad_inches=0.03, facecolor='white')
        print(f'Figure saved to {save_path}')
    plt.show()
    plt.close(fig)

In [ ]:
model_keys = ['clean', 'noisy', 'bm3d', 'dncnn', 'ffdnet', 'dranet', 'restormer', 'swinir', 'nafnet', 'admm']

In [ ]:
dset = 'BSD68_models_results_35_36std'
viz_img_paths = [f'D:/Projects/datasets/denoising_test_sets/gray/{dset}/{model_name}/2.png' for model_name in model_keys]
models_images = {model_key: Image.open(img_path) for model_key, img_path in zip(model_keys, viz_img_paths)}

In [ ]:
plot_image_comparison_grid(models_images, patch_size=256, crop_origin=(0, 0), nrows=2, ncols=5,
                           save_path=f'{dset}_comp.svg', dpi=300)

# Loss function sensitivity analysis (SSIM–DISTS weighting)

`SSIMDISTSLoss = w · (1 − SSIM) + (1 − w) · DISTS`. Starting from the same trained checkpoint, the model was fine-tuned for a few epochs with each weighting (`l1` = SSIM weight, `l2` = DISTS weight).

Because every run starts from the same checkpoint and sees the same data order, the per-epoch metrics of all runs fluctuate together (e.g. the common dip at epoch 3). To isolate the effect of the weighting, the comparison is **epoch-matched**: only epochs logged by every run are kept, and the epoch mean across runs is subtracted out (paired comparison). Error bars / bands are ±1 std of these paired values across epochs.

* **(a)** Eval perception–distortion frontier: PSNR (fidelity) vs DISTS (perceptual, x-axis inverted so right = better). Small dots are individual epochs.
* **(b)** Each metric normalized to [0, 1] across runs (1 = best run). Legend shows the raw range and the Spearman ρ between SSIM weight and metric *quality*.

In [ ]:
import re
from pathlib import Path

LOSS_RUNS_DIR = Path('D:/Projects/torch-admm-deconv/trained_models/2026/loss_analysis')


def load_loss_runs(root):
    """Load every l1-<ssim_w>_l2-<dists_w>/logged_metrics.csv into one long dataframe."""
    runs = []
    for csv_p in sorted(Path(root).glob('l1-*_l2-*/logged_metrics.csv')):
        match = re.match(r'l1-([\d.]+)_l2-([\d.]+)', csv_p.parent.name)
        df = pd.read_csv(csv_p, index_col=0).rename_axis('epoch').reset_index()
        df['ssim_w'], df['dists_w'] = float(match[1]), float(match[2])
        runs.append(df)
    return pd.concat(runs, ignore_index=True)


def epoch_matched(runs):
    """
    Keep only the epochs logged by every run and remove the epoch effect shared by all runs
    (same checkpoint init + same data order => all runs fluctuate together across epochs).
    value' = value - mean_over_runs(epoch) + grand_mean, so differences between runs are paired per epoch.
    """
    metric_cols = [c for c in runs.columns if c.startswith(('train_', 'eval_'))]
    n_runs = runs['ssim_w'].nunique()
    runs_per_epoch = runs.groupby('epoch')['ssim_w'].nunique()
    common_epochs = runs_per_epoch[runs_per_epoch == n_runs].index
    matched = runs[runs['epoch'].isin(common_epochs)].copy()
    matched[metric_cols] = (matched[metric_cols]
                            - matched.groupby('epoch')[metric_cols].transform('mean')
                            + matched[metric_cols].mean())
    return matched, list(common_epochs)


loss_runs = load_loss_runs(LOSS_RUNS_DIR)
loss_matched, common_epochs = epoch_matched(loss_runs)
loss_summary = loss_matched.groupby('ssim_w').agg(['mean', 'std'])
print(f"Runs (ssim_w): {sorted(loss_runs['ssim_w'].unique())}")
print(f"Epochs per run: {loss_runs.groupby('ssim_w')['epoch'].nunique().to_dict()}")
print(f"Common epochs used for the comparison: {common_epochs}")
loss_summary.xs('mean', axis=1, level=1)[['eval_psnr', 'eval_ssim', 'eval_uiq', 'eval_dists']].round(4)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

# Diverging weight map: blue pole = DISTS-heavy (perceptual), red pole = SSIM-heavy (fidelity), gray = balanced
WEIGHT_CMAP = LinearSegmentedColormap.from_list(
    'dists_ssim', ['#104281', '#3987e5', '#9a9a94', '#e66767', '#b02a2a'])

# (column suffix, label, higher_is_better, unit, color, marker)
SENS_METRICS = [
    ('psnr', 'PSNR', True, ' dB', '#e34948', 'o'),
    ('ssim', 'SSIM', True, '', '#eb6834', 's'),
    ('uiq', 'UIQ', True, '', '#6250d6', '^'),
    ('dists', 'DISTS', False, '', '#2a78d6', 'D'),
]


def _spearman(x, y):
    return pd.Series(x).rank().corr(pd.Series(y).rank())


def plot_loss_weight_sensitivity(matched, summary, epochs, selected_ssim_w=0.8, save_path=None):
    weights = summary.index.values
    mean = summary.xs('mean', axis=1, level=1)
    std = summary.xs('std', axis=1, level=1)
    norm = Normalize(0.0, 1.0)

    fig, (ax_f, ax_r) = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw={'width_ratios': [1.15, 1], 'wspace': 0.32})

    # ---------------- (a) perception-distortion frontier ----------------
    ax_f.plot(mean['eval_dists'], mean['eval_psnr'], color='#c3c2b7', lw=1.5, zorder=1)
    ax_f.scatter(matched['eval_dists'], matched['eval_psnr'], c=matched['ssim_w'], cmap=WEIGHT_CMAP, norm=norm,
                 s=14, alpha=0.3, linewidths=0, zorder=2)
    for w in weights:
        color = WEIGHT_CMAP(norm(w))
        ax_f.errorbar(mean.loc[w, 'eval_dists'], mean.loc[w, 'eval_psnr'],
                      xerr=std.loc[w, 'eval_dists'], yerr=std.loc[w, 'eval_psnr'],
                      fmt='none', ecolor=color, elinewidth=1.2, capsize=3, alpha=0.8, zorder=3)
        is_sel = selected_ssim_w is not None and np.isclose(w, selected_ssim_w)
        ax_f.scatter(mean.loc[w, 'eval_dists'], mean.loc[w, 'eval_psnr'], color=color, s=170 if is_sel else 90,
                     marker='*' if is_sel else 'o', edgecolor='white', linewidth=1.5, zorder=4)
        ax_f.annotate(f'{w:.1f}/{1 - w:.1f}', (mean.loc[w, 'eval_dists'], mean.loc[w, 'eval_psnr']),
                      xytext=(-9, 4), textcoords='offset points', ha='right', fontsize=9, color='#3d3d3a')

    # endpoint "exchange rate" between the most perceptual and most fidelity-oriented runs
    w_lo, w_hi = weights.min(), weights.max()
    d = mean.loc[w_hi] - mean.loc[w_lo]
    rel_dists = 100 * d['eval_dists'] / mean.loc[w_lo, 'eval_dists']
    ax_f.text(0.03, 0.04,
              f'SSIM weight {w_lo:.1f} → {w_hi:.1f}\n'
              f'ΔPSNR  {d["eval_psnr"]:+.3f} dB\n'
              f'ΔSSIM  {d["eval_ssim"]:+.4f}\n'
              f'ΔDISTS {d["eval_dists"]:+.4f} ({rel_dists:+.1f}%)',
              transform=ax_f.transAxes, va='bottom', ha='left', fontsize=9.5, family='monospace',
              bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#e1e0d9'))

    ax_f.invert_xaxis()  # lower DISTS is better -> better perception to the right
    ax_f.set_xlabel('Eval DISTS  (← worse   ·   better perceptual quality →)', fontsize=11)
    ax_f.set_ylabel('Eval PSNR (dB)  ↑ better fidelity', fontsize=11)
    ax_f.set_title('(a) Perception–distortion frontier', fontsize=13, fontweight='bold', loc='left')
    ax_f.grid(color='#e1e0d9', lw=0.6)
    sm = plt.cm.ScalarMappable(cmap=WEIGHT_CMAP, norm=norm)
    cbar = fig.colorbar(sm, ax=ax_f, pad=0.02, fraction=0.045)
    cbar.set_label('SSIM weight   (DISTS weight = 1 − SSIM weight)', fontsize=10)
    cbar.outline.set_visible(False)

    # ---------------- (b) normalized response of every metric ----------------
    for key, label, higher_better, unit, color, marker in SENS_METRICS:
        for split, ls, lw, alpha in [('eval', '-', 2.0, 1.0), ('train', '--', 1.2, 0.55)]:
            col = f'{split}_{key}'
            mu, sd = mean[col].values, std[col].values
            best, worst = (mu.max(), mu.min()) if higher_better else (mu.min(), mu.max())
            scale = best - worst
            score, band = (mu - worst) / scale, sd / abs(scale)
            if split == 'eval':
                # rank correlation between SSIM weight and metric *quality* (sign flipped for lower-is-better)
                rho = _spearman(weights, mu if higher_better else -mu)
                lbl = f'{label} {"↑" if higher_better else "↓"}  [{mu.min():.4g}–{mu.max():.4g}{unit}]  ρ={rho:+.2f}'
                ax_r.fill_between(weights, score - band, score + band, color=color, alpha=0.12, linewidth=0)
                ax_r.plot(weights, score, ls=ls, lw=lw, color=color, marker=marker, ms=7,
                          markeredgecolor='white', markeredgewidth=1.2, label=lbl)
            else:
                ax_r.plot(weights, score, ls=ls, lw=lw, color=color, alpha=alpha)

    if selected_ssim_w is not None:
        ax_r.axvline(selected_ssim_w, color='#9a9a94', lw=1, ls=':', zorder=0)
        ax_r.text(selected_ssim_w, 1.1, f' selected\n {selected_ssim_w:.1f}/{1 - selected_ssim_w:.1f}',
                  fontsize=8.5, color='#6b6b66', va='top', ha='right')
    ticks = np.round(np.arange(0, 1.01, 0.1), 1)
    ax_r.set_xlim(-0.03, 1.03)
    ax_r.set_ylim(-0.12, 1.12)
    ax_r.set_xticks(ticks, [f'{w:.1f}' for w in ticks], fontsize=9)
    # second tick row with the complementary DISTS weight
    ax_dists = ax_r.secondary_xaxis(-0.075)
    ax_dists.set_xticks(ticks, [f'{1 - w:.1f}' for w in ticks], fontsize=9)
    ax_dists.tick_params(length=0)
    ax_dists.spines['bottom'].set_visible(False)
    ax_dists.set_xlabel('Loss weights λ', fontsize=11)
    ax_r.set_ylabel('Normalized score  (1 = best run, 0 = worst run)', fontsize=11)
    ax_r.set_title('(b) Metric response to the loss weighting', fontsize=13, fontweight='bold', loc='left')
    ax_r.grid(color='#e1e0d9', lw=0.6)

    handles, labels = ax_r.get_legend_handles_labels()
    handles += [Line2D([], [], color='#3d3d3a', lw=2.0, ls='-'), Line2D([], [], color='#3d3d3a', lw=1.2, ls='--', alpha=0.55),
                Line2D([], [], color='none')]
    labels += ['eval (±1 std, epoch-matched)', 'train (scaled to its own best/worst)',
               f'ρ: Spearman correlation with SSIM λ (n = {len(weights)} runs)']
    ax_r.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9, frameon=False)

    for ax in (ax_f, ax_r):
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)

    fig.suptitle(f'SSIM–DISTS loss weighting sensitivity',
                 fontsize=12, y=1.0, color='#3d3d3a')

    # boxed "SSIM λ" / "DISTS λ" row labels, colored like the weight-map poles, enclosing each tick row
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_axes = ax_r.transAxes.inverted()
    for tick_ax, name, edge, face in [(ax_r, 'SSIM λ', '#b02a2a', '#fbe9e9'),
                                      (ax_dists, 'DISTS λ', '#104281', '#e3eefb')]:
        bbs = [t.get_window_extent(renderer) for t in tick_ax.get_xticklabels()]
        (x0, y0), (x1, y1) = to_axes.transform([(min(b.x0 for b in bbs), min(b.y0 for b in bbs)),
                                                (max(b.x1 for b in bbs), max(b.y1 for b in bbs))])
        label = ax_r.text(x0 - 0.03, (y0 + y1) / 2, name, transform=ax_r.transAxes, ha='right', va='center',
                          fontsize=9, fontweight='bold', color='#3d3d3a', clip_on=False)
        lx0 = to_axes.transform((label.get_window_extent(renderer).x0, 0))[0]
        pad_x, pad_y = 0.012, 0.006
        ax_r.add_patch(FancyBboxPatch((lx0 - pad_x, y0 - pad_y), (x1 - lx0) + 2 * pad_x, (y1 - y0) + 2 * pad_y,
                                      boxstyle='round,pad=0,rounding_size=0.012', transform=ax_r.transAxes,
                                      facecolor=face, edgecolor=edge, linewidth=1.1, clip_on=False, zorder=-1))

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


plot_loss_weight_sensitivity(loss_matched, loss_summary, common_epochs, selected_ssim_w=0.8,
                             save_path='loss_weight_sensitivity.png')